In [5]:
import torch
import cv2
import numpy as np
import pickle
from PIL import Image
from facenet_pytorch import MTCNN, InceptionResnetV1

# --- CONFIG (Same as training) ---
MODEL_SAVE_PATH = "model5.pkl"
CONFIDENCE_THRESHOLD = 0.55
IMAGE_SIZE = (160, 160)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1. MODEL DEFINITIONS (Wahi class jo pehle thi) ---
class DeconvSkipHead(torch.nn.Module):
    def __init__(self, embed_dim=512):
        super().__init__()
        self.fc_in = torch.nn.Linear(embed_dim, 32 * 4 * 4)
        self.bn_in = torch.nn.BatchNorm1d(32 * 4 * 4)
        self.deconv1 = torch.nn.Sequential(
            torch.nn.ConvTranspose2d(32, 64, kernel_size=4, stride=2, padding=1),
            torch.nn.BatchNorm2d(64),
            torch.nn.ReLU(inplace=True),
        )
        self.deconv2 = torch.nn.Sequential(
            torch.nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            torch.nn.BatchNorm2d(32),
            torch.nn.ReLU(inplace=True),
        )
        self.skip_up = torch.nn.Upsample(size=(16, 16), mode="bilinear", align_corners=False)
        self.merge = torch.nn.Sequential(
            torch.nn.Conv2d(64, 32, kernel_size=1),
            torch.nn.BatchNorm2d(32),
            torch.nn.ReLU(inplace=True),
        )
        self.pool = torch.nn.AdaptiveAvgPool2d(1)
        self.fc_out = torch.nn.Linear(32, embed_dim)
        self.gate = torch.nn.Parameter(torch.tensor(0.1))

    def forward(self, x):
        identity = x
        h = torch.relu(self.bn_in(self.fc_in(x)))
        h = h.view(-1, 32, 4, 4)
        skip = h
        h = self.deconv1(h)
        h = self.deconv2(h)
        h = self.merge(torch.cat([h, self.skip_up(skip)], dim=1))
        h = self.pool(h).flatten(1)
        h = self.fc_out(h)
        return identity + self.gate * h

# --- 2. LOAD EVERYTHING ---
print("[INFO] Loading pre-trained FaceNet and your model3.pkl...")
mtcnn = MTCNN(image_size=160, margin=20, keep_all=False, device=DEVICE)
resnet = InceptionResnetV1(pretrained="vggface2").eval().to(DEVICE)
head = DeconvSkipHead(embed_dim=512).eval().to(DEVICE)

with open(MODEL_SAVE_PATH, "rb") as f:
    data = pickle.load(f)
    clf = data["clf"]
    le = data["le"]

# --- 3. WEBCAM PREDICTION ---
cap = cv2.VideoCapture(0)
print("[LIVE] Webcam started. Press 'q' to quit.")

while True:
    ret, frame = cap.read()
    if not ret: break

    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img_rgb)
    
    # Detect face
    boxes, _ = mtcnn.detect(pil_img)

    if boxes is not None:
        for box in boxes:
            x1, y1, x2, y2 = [int(v) for v in box]
            face = img_rgb[max(0,y1):y2, max(0,x1):x2]
            
            if face.size > 0:
                face_pil = Image.fromarray(face).resize(IMAGE_SIZE)
                face_tensor = mtcnn(face_pil)
                
                if face_tensor is not None:
                    face_tensor = face_tensor.unsqueeze(0).to(DEVICE)
                    with torch.no_grad():
                        emb = head(resnet(face_tensor)).squeeze().cpu().numpy()
                    
                    # Predict using SVM
                    probs = clf.predict_proba([emb])[0]
                    max_prob = np.max(probs)
                    label = le.inverse_transform([np.argmax(probs)])[0] if max_prob > CONFIDENCE_THRESHOLD else "Unknown"

                    # Draw
                    color = (0, 255, 0) if label != "Unknown" else (0, 0, 255)
                    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                    cv2.putText(frame, f"{label} ({round(max_prob*100,1)}%)", (x1, y1-10), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    cv2.imshow("Prediction Mode", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

[INFO] Loading pre-trained FaceNet and your model3.pkl...
[LIVE] Webcam started. Press 'q' to quit.


In [1]:
import torch
import torch.nn as nn
import cv2
import numpy as np
import pickle
from PIL import Image
from facenet_pytorch import MTCNN, InceptionResnetV1

# --- CONFIG ---
MODEL_SAVE_PATH = "model4.pkl"
CONFIDENCE_THRESHOLD = 0.10
IMAGE_SIZE = (160, 160)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─────────────────────────────────────────────────────────────
# NEW TECHNIQUE: FLATTEN + 3 FULLY CONNECTED LAYERS (DENSE HEAD)
# ─────────────────────────────────────────────────────────────
class FlattenDenseHead(nn.Module):
    """
    Yahan humne DeconvSkipHead ki jagah 3 Deep Fully Connected layers lagayi hain.
    Yeh FaceNet ke features ko deeply analyze karke refine karta hai.
    """
    def __init__(self, embed_dim=512):
        super().__init__()
        
        # [NEW] Flatten layer manually implement karne ki zaroorat nahi 
        # kyunki hum forward pass mein view() use karenge.

        # Layer 1: Input 512 -> 1024
        self.fc1 = nn.Linear(embed_dim, 1024)
        self.bn1 = nn.BatchNorm1d(1024)
        self.dropout1 = nn.Dropout(0.3) # Overfitting se bachne ke liye
        
        # Layer 2: 1024 -> 512
        self.fc2 = nn.Linear(1024, 512)
        self.bn2 = nn.BatchNorm1d(512)
        
        # Layer 3: 512 -> 512 (Final Output for SVM)
        self.fc3 = nn.Linear(512, embed_dim)
        
        self.relu = nn.ReLU()

    def forward(self, x):
        # [CHANGE] FaceNet ka output flatten karke fully connected layers mein bhej rahe hain
        x = x.view(x.size(0), -1) 
        
        # Fully Connected Layer 1
        x = self.relu(self.bn1(self.fc1(x)))
        x = self.dropout1(x)
        
        # Fully Connected Layer 2
        x = self.relu(self.bn2(self.fc2(x)))
        
        # Fully Connected Layer 3 (Final refined embedding)
        x = self.fc3(x)
        return x

# --- 2. LOAD EVERYTHING ---
print("[INFO] Loading FaceNet + New Flatten-Dense Technique...")
mtcnn = MTCNN(image_size=160, margin=20, keep_all=False, device=DEVICE)
resnet = InceptionResnetV1(pretrained="vggface2").eval().to(DEVICE)

# [CHANGE] Ab humara 'head' naya FlattenDenseHead use karega
head = FlattenDenseHead(embed_dim=512).eval().to(DEVICE)

# Model load karna
with open(MODEL_SAVE_PATH, "rb") as f:
    data = pickle.load(f)
    clf = data["clf"]
    le = data["le"]

# --- 3. WEBCAM PREDICTION ---
cap = cv2.VideoCapture(0)
print("[LIVE] Webcam started. Press 'q' to quit.")

while True:
    ret, frame = cap.read()
    if not ret: break

    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img_rgb)
    
    # Detect face
    boxes, _ = mtcnn.detect(pil_img)

    if boxes is not None:
        for box in boxes:
            x1, y1, x2, y2 = [int(v) for v in box]
            face = img_rgb[max(0,y1):y2, max(0,x1):x2]
            
            if face.size > 0:
                face_pil = Image.fromarray(face).resize(IMAGE_SIZE)
                face_tensor = mtcnn(face_pil)
                
                if face_tensor is not None:
                    face_tensor = face_tensor.unsqueeze(0).to(DEVICE)
                    with torch.no_grad():
                        # [CHANGE] Refinement ab naye Dense layers se ho rahi hai
                        raw_emb = resnet(face_tensor)
                        refined_emb = head(raw_emb) 
                        emb = refined_emb.squeeze().cpu().numpy()
                    
                    # Predict using SVM
                    probs = clf.predict_proba([emb])[0]
                    max_prob = np.max(probs)
                    label = le.inverse_transform([np.argmax(probs)])[0] if max_prob > CONFIDENCE_THRESHOLD else "Unknown"

                    # Drawing logic
                    color = (0, 255, 0) if label != "Unknown" else (0, 0, 255)
                    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                    cv2.putText(frame, f"{label} ({round(max_prob*100,1)}%)", (x1, y1-10), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    cv2.imshow("Prediction Mode (Flatten-Dense)", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

[INFO] Loading FaceNet + New Flatten-Dense Technique...
[LIVE] Webcam started. Press 'q' to quit.


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import cv2
import numpy as np
import pickle
from PIL import Image
from facenet_pytorch import MTCNN, InceptionResnetV1
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

# --- CONFIG ---
FACES_DIR = "known_faces_folder"  # Aapke photos ka folder yahan hona chahiye
MODEL_SAVE_PATH = "model4.pkl"
IMAGE_SIZE = (160, 160)
BATCH_SIZE = 16
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─────────────────────────────────────────────────────────────
# 1. NEW ARCHITECTURE: FLATTEN + DENSE HEAD (RNN Style Layers)
# ─────────────────────────────────────────────────────────────
class FlattenDenseHead(nn.Module):
    """
    Yeh FaceNet ke output ko flatten karke 3 deep layers se guzaarta hai.
    """
    def __init__(self, embed_dim=512):
        super().__init__()
        # Layer 1: Expand Features
        self.fc1 = nn.Linear(embed_dim, 1024)
        self.bn1 = nn.BatchNorm1d(1024)
        self.dropout1 = nn.Dropout(0.3)
        
        # Layer 2: Deep Processing
        self.fc2 = nn.Linear(1024, 512)
        self.bn2 = nn.BatchNorm1d(512)
        
        # Layer 3: Final Refined Vector
        self.fc3 = nn.Linear(512, embed_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        # [FLATTEN] Input ko flat kar raha hai
        x = x.view(x.size(0), -1) 
        x = self.relu(self.bn1(self.fc1(x)))
        x = self.dropout1(x)
        x = self.relu(self.bn2(self.fc2(x)))
        x = self.fc3(x)
        return x

# --- 2. MODELS INITIALIZATION ---
print(f"[INFO] Using Device: {DEVICE}")
mtcnn = MTCNN(image_size=160, margin=20, keep_all=False, device=DEVICE)
resnet = InceptionResnetV1(pretrained="vggface2").eval().to(DEVICE)
head = FlattenDenseHead(embed_dim=512).eval().to(DEVICE) # Naya Head

# --- 3. TRAINING FUNCTION ---
def train_model():
    print("[TRAIN] Starting Training Workflow...")
    
    embeddings = []
    labels = []
    
    # Dataset scan kar rahe hain
    for person_name in os.listdir(FACES_DIR):
        person_path = os.path.join(FACES_DIR, person_name)
        if not os.path.isdir(person_path): continue
        
        print(f"[TRAIN] Processing: {person_name}")
        for img_name in os.listdir(person_path):
            img_path = os.path.join(person_path, img_name)
            try:
                img = Image.open(img_path).convert('RGB')
                # MTCNN se face nikalna
                face = mtcnn(img)
                if face is not None:
                    with torch.no_grad():
                        # FaceNet + Naya Head use karke embedding nikalna
                        raw_emb = resnet(face.unsqueeze(0).to(DEVICE))
                        refined_emb = head(raw_emb)
                        embeddings.append(refined_emb.squeeze().cpu().numpy())
                        labels.append(person_name)
            except Exception as e:
                print(f"Error skipping {img_name}: {e}")

    # Label Encoding (Naam ko numbers mein badalna)
    le = LabelEncoder()
    y = le.fit_transform(labels)
    X = np.array(embeddings)

    # SVM Classifier (Jo decide karega kaun kiska chehra hai)
    print("[TRAIN] Fitting SVM Classifier...")
    clf = SVC(kernel='linear', probability=True)
    clf.fit(X, y)

    # Save Model
    with open(MODEL_SAVE_PATH, "wb") as f:
        pickle.dump({"clf": clf, "le": le}, f)
    
    print(f"[SUCCESS] Model saved as {MODEL_SAVE_PATH}")
    return clf, le

# --- 4. PREDICTION (WEBCAM) ---
def start_webcam(clf, le):
    cap = cv2.VideoCapture(0)
    print("[LIVE] Press 'q' to quit.")
    
    while True:
        ret, frame = cap.read()
        if not ret: break
        
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(img_rgb)
        
        boxes, _ = mtcnn.detect(pil_img)
        if boxes is not None:
            for box in boxes:
                x1, y1, x2, y2 = [int(v) for v in box]
                face = img_rgb[max(0,y1):y2, max(0,x1):x2]
                
                if face.size > 0:
                    face_pil = Image.fromarray(face).resize(IMAGE_SIZE)
                    face_tensor = mtcnn(face_pil)
                    
                    if face_tensor is not None:
                        with torch.no_grad():
                            raw_emb = resnet(face_tensor.unsqueeze(0).to(DEVICE))
                            refined_emb = head(raw_emb)
                            emb = refined_emb.squeeze().cpu().numpy()
                        
                        probs = clf.predict_proba([emb])[0]
                        max_prob = np.max(probs)
                        name = le.inverse_transform([np.argmax(probs)])[0] if max_prob > 0.55 else "Unknown"
                        
                        color = (0, 255, 0) if name != "Unknown" else (0, 0, 255)
                        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                        cv2.putText(frame, f"{name} {round(max_prob*100,1)}%", (x1, y1-10), 
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
        
        cv2.imshow("New Flatten-Dense Model", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'): break
    
    cap.release()
    cv2.destroyAllWindows()

# --- EXECUTION ---
if __name__ == "__main__":
    # Pehle training hogi (naye architecture ke saath)
    classifier, label_encoder = train_model()
    
    # Phir webcam chalu hoga
    start_webcam(classifier, label_encoder)

[INFO] Using Device: cpu
[TRAIN] Starting Training Workflow...
[TRAIN] Processing: aditya joshi
[TRAIN] Processing: aditya kumar
[TRAIN] Processing: amit
[TRAIN] Processing: bhawna
[TRAIN] Processing: harsh
[TRAIN] Processing: harshit
[TRAIN] Processing: pranjal
[TRAIN] Processing: ravi
[TRAIN] Processing: rupesh_kattebaaz
[TRAIN] Processing: yadav
[TRAIN] Fitting SVM Classifier...
[SUCCESS] Model saved as model4.pkl
[LIVE] Press 'q' to quit.


In [3]:
import os
import torch
import cv2
import numpy as np
import pickle
from PIL import Image
from facenet_pytorch import MTCNN, InceptionResnetV1
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder

# --- CONFIG ---
FACES_DIR = "known_faces_folder"
MODEL_SAVE_PATH = "model4.pkl"
IMAGE_SIZE = (160, 160)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1. MODELS INITIALIZATION ---
print(f"[INFO] Using Device: {DEVICE}")
# Face Detection ke liye
mtcnn = MTCNN(image_size=160, margin=20, keep_all=False, device=DEVICE)
# Direct FaceNet (No extra layers/heads)
resnet = InceptionResnetV1(pretrained="vggface2").eval().to(DEVICE)

# --- 2. TRAINING FUNCTION (Only FaceNet) ---
def train_pure_facenet():
    print("[TRAIN] Starting Pure FaceNet Training...")
    embeddings = []
    labels = []
    
    if not os.path.exists(FACES_DIR):
        print(f"[ERROR] Folder '{FACES_DIR}' nahi mila!")
        return None, None

    for person_name in os.listdir(FACES_DIR):
        person_path = os.path.join(FACES_DIR, person_name)
        if not os.path.isdir(person_path): continue
        
        print(f"[TRAIN] Processing: {person_name}")
        for img_name in os.listdir(person_path):
            img_path = os.path.join(person_path, img_name)
            try:
                img = Image.open(img_path).convert('RGB')
                face = mtcnn(img) # Face detect and crop
                if face is not None:
                    with torch.no_grad():
                        # Direct FaceNet se embedding nikalna
                        emb = resnet(face.unsqueeze(0).to(DEVICE))
                        embeddings.append(emb.squeeze().cpu().numpy())
                        labels.append(person_name)
            except Exception as e:
                print(f"Error skipping {img_name}: {e}")

    if len(embeddings) == 0:
        print("[ERROR] Koi faces nahi mile! Training cancel.")
        return None, None

    # Label Encoding
    le = LabelEncoder()
    y = le.fit_transform(labels)
    X = np.array(embeddings)

    # SVM Classifier (Directly on FaceNet output)
    print("[TRAIN] Fitting SVM...")
    clf = SVC(kernel='linear', probability=True)
    clf.fit(X, y)

    # Save
    with open(MODEL_SAVE_PATH, "wb") as f:
        pickle.dump({"clf": clf, "le": le}, f)
    
    print(f"[SUCCESS] Model saved to {MODEL_SAVE_PATH}")
    return clf, le

# --- 3. PREDICTION (WEBCAM) ---
def start_webcam(clf, le):
    cap = cv2.VideoCapture(0)
    print("[LIVE] Webcam Started. Press 'q' to quit.")
    
    while True:
        ret, frame = cap.read()
        if not ret: break
        
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(img_rgb)
        
        boxes, _ = mtcnn.detect(pil_img)
        if boxes is not None:
            for box in boxes:
                x1, y1, x2, y2 = [int(v) for v in box]
                face_crop = img_rgb[max(0,y1):y2, max(0,x1):x2]
                
                if face_crop.size > 0:
                    face_pil = Image.fromarray(face_crop).resize(IMAGE_SIZE)
                    face_tensor = mtcnn(face_pil)
                    
                    if face_tensor is not None:
                        with torch.no_grad():
                            # Prediction mein bhi direct FaceNet
                            emb = resnet(face_tensor.unsqueeze(0).to(DEVICE))
                            emb_np = emb.squeeze().cpu().numpy()
                        
                        probs = clf.predict_proba([emb_np])[0]
                        max_prob = np.max(probs)
                        name = le.inverse_transform([np.argmax(probs)])[0] if max_prob > 0.60 else "Unknown"
                        
                        color = (0, 255, 0) if name != "Unknown" else (0, 0, 255)
                        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                        cv2.putText(frame, f"{name} {round(max_prob*100,1)}%", (x1, y1-10), 
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
        
        cv2.imshow("Pure FaceNet Mode", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'): break
    
    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    # Nayi training sirf FaceNet ke saath
    classifier, label_encoder = train_pure_facenet()
    
    if classifier:
        start_webcam(classifier, label_encoder)

[INFO] Using Device: cpu
[TRAIN] Starting Pure FaceNet Training...
[TRAIN] Processing: aditya joshi
[TRAIN] Processing: aditya kumar
[TRAIN] Processing: amit
[TRAIN] Processing: bhawna
[TRAIN] Processing: harsh
[TRAIN] Processing: harshit
[TRAIN] Processing: pranjal
[TRAIN] Processing: ravi
[TRAIN] Processing: rupesh_kattebaaz
[TRAIN] Processing: yadav
[TRAIN] Fitting SVM...
[SUCCESS] Model saved to model4.pkl
[LIVE] Webcam Started. Press 'q' to quit.


In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import cv2
import numpy as np
import pickle
from PIL import Image
from facenet_pytorch import MTCNN, InceptionResnetV1
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder

# --- CONFIG ---
FACES_DIR = "known_faces_folder"
MODEL_SAVE_PATH = "model2.pkl"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─────────────────────────────────────────────────────────────
# NEW TECHNIQUE: ATTENTION-BASED RESIDUAL HEAD
# ─────────────────────────────────────────────────────────────
class AttentionResidualHead(nn.Module):
    def __init__(self, embed_dim=512):
        super().__init__()
        # 1. SE Block (Attention): Kaunse features important hain?
        self.attention = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 16),
            nn.ReLU(),
            nn.Linear(embed_dim // 16, embed_dim),
            nn.Sigmoid()
        )
        
        # 2. Refinement Layers
        self.fc = nn.Sequential(
            nn.Linear(embed_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(1024, embed_dim)
        )

    def forward(self, x):
        identity = x # Original FaceNet embedding
        
        # Apply Attention
        att_weights = self.attention(x)
        x = x * att_weights
        
        # Refine through FC layers
        x = self.fc(x)
        
        # Residual Connection: Purana + Naya (Impact yahi se aata hai)
        return identity + x

# --- INITIALIZATION ---
mtcnn = MTCNN(image_size=160, margin=20, device=DEVICE)
resnet = InceptionResnetV1(pretrained="vggface2").eval().to(DEVICE)
head = AttentionResidualHead(embed_dim=512).eval().to(DEVICE)

# --- TRAINING WITH ATTENTION ---
def train_with_attention():
    print("[TRAIN] Starting Attention-Residual Training...")
    embeddings, labels = [], []

    for person_name in os.listdir(FACES_DIR):
        person_path = os.path.join(FACES_DIR, person_name)
        if not os.path.isdir(person_path): continue
        
        for img_name in os.listdir(person_path):
            try:
                img = Image.open(os.path.join(person_path, img_name)).convert('RGB')
                face = mtcnn(img)
                if face is not None:
                    with torch.no_grad():
                        raw_emb = resnet(face.unsqueeze(0).to(DEVICE))
                        refined_emb = head(raw_emb) # Passing through Attention Head
                        embeddings.append(refined_emb.squeeze().cpu().numpy())
                        labels.append(person_name)
            except: continue

    le = LabelEncoder()
    y = le.fit_transform(labels)
    X = np.array(embeddings)

    # Higher C value for SVM to handle complex features
    clf = SVC(kernel='rbf', probability=True, C=10) 
    clf.fit(X, y)

    with open(MODEL_SAVE_PATH, "wb") as f:
        pickle.dump({"clf": clf, "le": le}, f)
    print("[SUCCESS] Model trained with Attention Head!")
    return clf, le

# --- LIVE PREDICTION ---
def start_webcam(clf, le):
    cap = cv2.VideoCapture(0)
    while True:
        ret, frame = cap.read()
        if not ret: break
        
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(img_rgb)
        boxes, _ = mtcnn.detect(pil_img)

        if boxes is not None:
            for box in boxes:
                x1, y1, x2, y2 = [int(v) for v in box]
                face_tensor = mtcnn(Image.fromarray(img_rgb[max(0,y1):y2, max(0,x1):x2]).resize((160,160)))
                
                if face_tensor is not None:
                    with torch.no_grad():
                        emb = head(resnet(face_tensor.unsqueeze(0).to(DEVICE))).squeeze().cpu().numpy()
                    
                    probs = clf.predict_proba([emb])[0]
                    name = le.inverse_transform([np.argmax(probs)])[0] if np.max(probs) > 0.55 else "Unknown"
                    
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    cv2.putText(frame, f"{name} {round(np.max(probs)*100,1)}%", (x1, y1-10), 0, 0.7, (0,255,0), 2)

        cv2.imshow("Attention-Residual Recognition", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'): break
    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    c, l = train_with_attention()
    if c: start_webcam(c, l)

[TRAIN] Starting Attention-Residual Training...
[SUCCESS] Model trained with Attention Head!
